**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/alonso-2020"], check=True)
    os.chdir("/content/encore/entries/alonso-2020")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Alonso & Marder (2020) from Dryad m31b1h7

Paper: Alonso LM, Marder E (2020). Temperature compensation in a small rhythmic circuit. eLife
9:e55470, doi:10.7554/eLife.55470. The claims file, `claims/alonso-2020.yaml`, was written from
the version of record (main text, legends and Methods; the figure supplements' legends were not
in the text read).

Dataset: Dryad doi:10.5061/dryad.m31b1h7, version id 94811, one file,
`alonso_marder_tempcomp_2019_root.zip` (99,026 bytes, MD5 37766522dabaaa620d4c44264e76e686 from
Dryad's file list; its SHA-256 is printed below). Version 94811 is pinned because it is the only
published version of the dataset (the versions list of the API gives one, published 2020-12-07),
and the paper cites the dataset by its DOI, which resolves to it. Dryad needs a token to
download, so the workflow stages the file before the run and the notebook reads it in place,
into memory (docs/archives.md, "Dryad").

The paper has no recordings. The dataset holds the authors' Java model, Python driver scripts,
initial conditions and the parameters of each model network. Every claim is tested by simulating
the deposited models with a numba port of the deposited Java (`pyloric.py`), checked against the
Java below when a Java compiler is present.

This is the first of three notebooks. `ramps.ipynb` tests the claims on temperature ramps
(figs2-ramp-largest-isi, fig8a-hysteresis-quiescence, fig8-up-down-ramps,
fig8b-pathological-attractor) and `voltage_distributions.ipynb` the claims
on membrane potential distributions (fig2-lp-spike-amplitude-threshold,
fig2s1-peak-voltage-diverse).

| Claim | Notebook | Tested |
|---|---|---|
| deposit-models | this | tested |
| fig1c-frequency-q10-phase-constant | this | tested |
| fig1s1-three-models | this | tested |
| all-models-compensated | this | tested |
| intermediate-temperature-failures | this | blocked (data_missing) |
| figs2-ramp-largest-isi | ramps | tested |
| figs3-duty-cycle-variable-across-models | this | tested (variant reading) |
| fig2-lp-spike-amplitude-threshold | voltage_distributions | tested (threshold reading b assumed) |
| fig2s1-peak-voltage-diverse | voltage_distributions | tested |
| fig3-current-amplitudes | this | tested |
| fig4-currentscapes-reorganize | this | tested |
| currentscapes-dominant-currents | this | tested |
| fig5-a-to-kca-tradeoff | this | tested |
| fig6-crash-different-mechanisms | this | tested |
| fig6s1-crash-mechanisms | this | blocked (data_missing; 7 of 8 models tested) |
| fig7-high-temperature-slow-timescales | this | blocked (data_missing, 3 of 5 models absent) |
| fig8a-hysteresis-quiescence | ramps | tested |
| fig8-up-down-ramps | ramps | blocked (too_large); first reading shown |
| fig8b-pathological-attractor | ramps | tested |
| fig9a-remove-a-current | this | tested |
| fig9b-remove-kca-current | this | tested |
| fig10a-graded-removal | this | tested |
| fig10b-complete-removal-differs | this | tested |

In [ ]:
import os
import re
import resource
import shutil
import subprocess
import tempfile
import time
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

import pyloric as P

T_START = time.time()
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

raw = P.read_zip_bytes()
DIG = P.digests(raw)
Z = P.open_zip(raw)
MODELS = P.load_models(Z)
FILES = P.model_files(Z)
CI = P.load_ci(Z)
NAMES = list(MODELS)
print(DIG)

## Data check

In [ ]:
members = [i for i in Z.infolist() if not i.filename.startswith("__MACOSX/")]
print(f"{len(Z.infolist())} members in the zip, {len(members)} outside __MACOSX/")
for i in members:
    print(f"  {i.filename:110s} {i.file_size:7d}")
print()
print("parameter files:", len(FILES), "models")
lengths = Counter(len(p) for p in MODELS.values())
print("numbers per parameter file:", dict(lengths), "(3 cells x 37 + 7 synaptic g + 14 synaptic Q10 = 132)")
print("initial conditions:", len(CI), "numbers; per cell V, 11 gates, [Ca]:")
for c in range(3):
    print("  ", P.CELLS[c], CI[13 * c:13 * c + 13])
print("   synapses", CI[39:])
G = pd.DataFrame({n: P.conductances(MODELS[n]).ravel() for n in NAMES},
                 index=[f"{c} g{k}" for c in P.CELLS for k in P.CURRENTS]).T
print("\nmaximal conductances (uS), range over models:")
print(G.agg(["min", "max"]).T.to_string())
tauca = pd.DataFrame({n: MODELS[n][[35, 72, 109]] * 10 for n in NAMES}, index=["PD", "LP", "PY"]).T
print("\ntau_Ca (ms; the files hold tau_Ca / 10 and the Java multiplies by 10):")
print(tauca.agg(["min", "max"]).to_string())
print("\nsynaptic conductances (uS), range:", np.round(np.min([MODELS[n][111:118] for n in NAMES], 0), 4),
      np.round(np.max([MODELS[n][111:118] for n in NAMES], 0), 4))
rev = {tuple(MODELS[n][37 * c + 8:37 * c + 12]) for n in NAMES for c in range(3)}
print("reversal potentials (EL, ENa, EK, EH), all cells and models:", rev)
print("injected current, all cells and models:", {MODELS[n][37 * c + 36] for n in NAMES for c in range(3)})
same_q = all(np.array_equal(MODELS[n][12:35], MODELS[n][49:72]) and np.array_equal(MODELS[n][12:35], MODELS[n][86:109])
             for n in NAMES)
print("Q10 values identical in the three cells of every model:", same_q)

The port. The deposit's Java is compiled into a temporary directory (outside the dataset) and
run for 3 s at 18 degrees C on one model, when a Java compiler is installed; the port's voltage
traces are compared with its output. It is also compared above 18 degrees C: J9U8SQ for 3 s at 25
degrees C, and the two runs whose integration diverges in the port, G7P2WE at 35 degrees C and
J9U8SQ at 34 degrees C, over their first second (the Java takes whole seconds).

In [ ]:
def java_check(name="4X6W66", seconds=3, temp=18):
    """Runs the deposited Java and the port from the deposited initial conditions; returns the largest
    difference in V over the samples where both are finite, and the first sample with NaN in each."""
    if not (shutil.which("javac") and shutil.which("java")):
        print("no Java compiler here; comparison not run")
        return None
    with tempfile.TemporaryDirectory() as d:
        src = [m for m in Z.namelist() if m.startswith(P.ROOT + "standaloneCodePyloricTempComp/") and m.endswith(".java")]
        pkg = Path(d) / "standaloneCodePyloricTempComp"
        pkg.mkdir()
        for m in src:
            (pkg / Path(m).name).write_bytes(Z.read(m))
        (Path(d) / "ci.txt").write_bytes(Z.read(P.ROOT + "parameters_and_ci/cis_network.txt"))
        (Path(d) / "p.txt").write_bytes(Z.read(FILES[name]))
        subprocess.run(["javac"] + [str(pkg / Path(m).name) for m in src], check=True, cwd=d)
        out = subprocess.run(["java", "-cp", d, "standaloneCodePyloricTempComp.integratePyloricNetwork_slow_full_temp_sameq10s_rk4",
                              "ci.txt", "p.txt", str(seconds), str(temp), "0.05"], cwd=d, capture_output=True,
                             text=True, check=True).stdout
    j = np.array([[float(x) for x in line.split()] for line in out.strip().splitlines()])
    s = P.simulate(MODELS[name], temp, seconds, CI, rec_every=1)
    n = min(len(j), len(s.rec))
    both = np.isfinite(j[:n]).all(axis=1) & np.isfinite(s.rec[:n]).all(axis=1)
    first_nan = [int(np.argmax(~np.isfinite(x[:n]).all(axis=1))) if (~np.isfinite(x[:n]).all(axis=1)).any() else None for x in (j, s.rec)]
    diff = np.abs(j[:n][both] - s.rec[:n][both]).max()
    print(f"{name}, {seconds} s at {temp} C: Java {j.shape}, port {s.rec.shape}; largest difference in V where both are finite "
          f"{diff:.2e} mV over {int(both.sum())} samples; first sample with NaN: Java {first_nan[0]}, port {first_nan[1]}")
    return diff

java_diff = java_check()
# Above the working range: a model that stays finite at 25 C, and the two runs that diverge in the port (Figure 7A shows
# J9U8SQ active at 34 C).
java_hot = {k: java_check(*k) for k in [("J9U8SQ", 3, 25), ("G7P2WE", 1, 35), ("J9U8SQ", 1, 34)]}

### Model names: the Methods' list against the deposit

The Methods (Parameters used in this study) name 21 models. The deposit's file names are
compared with them exactly, and then allowing the letter O for the digit 0 and a swap of two
neighbouring characters.

In [ ]:
METHODS_NAMES = {
    "Figure 1": ["4?6W66"], "Figure 1-S1": ["WWZ3CN", "FGFKQS", "71G6LA"], "Figure 1-S2": ["PLKTEM"],
    "Figure 2": ["G7P2WE"], "Figure 2-S1": ["G7P2WE", "4?6W66", "MAXTTP"], "Figure 3": ["4?6W66", "SJR46Y"],
    "Figure 4": ["WWZ3CN"], "Figure 5": ["FGFKQS"], "Figure 6": ["FGFKQS", "71G6LA"],
    "Figure 6-S1": ["B7SDOL", "J9U8SQ", "4R2719", "R3VIEW", "Q8FBN1", "RMZ8KZ", "EF84RN", "SJR46Y"],
    "Figure 7": ["J9U8SQ", "G81PUL", "G7EOZ8", "SAXT8Y", "KNMLBC"], "Figure 8": ["OSHNMD", "H5AIZR"],
    "Figure 9": ["WWZ3CN", "FGFKQS"], "Figure 10": ["4?6W66"],
}

def norm(s):
    return s.replace("O", "0")

def match(name):
    if name in MODELS:
        return name, "exact"
    pat = re.compile("^" + norm(name).replace("?", ".") + "$")
    hits = [n for n in NAMES if pat.match(norm(n))]
    if len(hits) == 1:
        return hits[0], "O read as 0" if "?" not in name else "unreadable character matched"
    for n in NAMES:
        a, b = norm(name), norm(n)
        d = [i for i in range(6) if a[i] != b[i]]
        if len(d) == 2 and d[1] == d[0] + 1 and a[d[0]] == b[d[1]] and a[d[1]] == b[d[0]]:
            return n, "two characters swapped"
    return None, "not in the deposit"

uniq = sorted({n for v in METHODS_NAMES.values() for n in v})
name_map = pd.DataFrame([(n, *match(n), ", ".join(k for k, v in METHODS_NAMES.items() if n in v)) for n in uniq],
                        columns=["Methods", "deposit", "how", "figures"])
print(name_map.to_string(index=False))
M = {r.Methods: r.deposit for r in name_map.itertuples()}
print("\nmodels in the deposit:", len(NAMES), "; named in the Methods and found:", name_map.deposit.notna().sum(),
      "of", len(name_map))

In [ ]:
# Four models the Methods name are not in the deposit, and four others are filed under other spellings.
print("deposit:", len(P.model_files(Z)), "parameter files;", sorted(P.model_files(Z)))
for n in ["Q8FBN1", "G81PUL", "G7EOZ8", "SAXT8Y", "OSHNMD", "B7SDOL", "KNMLBC"]:
    print(f"  {n}: in deposit {n in P.model_files(Z)}")

### Models with the same maximal conductances

The Results say 36 sets of maximal conductances were selected and Q10 values searched for each.
Models whose 31 conductances and three tau_Ca are identical are grouped here.

In [ ]:
key = {n: tuple(np.r_[P.conductances(MODELS[n]).ravel(), MODELS[n][[35, 72, 109]], MODELS[n][111:118]]) for n in NAMES}
groups = {}
for n in NAMES:
    groups.setdefault(key[n], []).append(n)
multi = [g for g in groups.values() if len(g) > 1]
print(f"{len(NAMES)} models, {len(groups)} distinct sets of maximal conductances; "
      f"{len(multi)} sets carry more than one Q10 set:")
for g in multi:
    q = np.array([MODELS[n][12:35] for n in g])
    print(f"  {g}: Q10 sets differ: {not np.all(q == q[0])}")

### Parameters against the Methods' search space

The Methods (Finding parameters) list the search space: gNa and gKCa in [0, 10^3] uS, gCaT, gCaS,
gA, gKd and gH in [0, 10^2], gleak in [0, 10], tau_Ca in [0, 2 x 10^3] ms and every synaptic
conductance in [0, 5 x 10^-2] uS. The deposited values are compared with these bounds, tau_Ca both as
stored in the files and as the Java uses it (times 10).

In [ ]:
# Deposited parameters against the search space of the Methods (Finding parameters).
BOUNDS = {"Na": 1e3, "CaT": 1e2, "CaS": 1e2, "A": 1e2, "KCa": 1e3, "Kd": 1e2, "H": 1e2, "Leak": 10}
for k, cur in enumerate(P.CURRENTS):
    v = np.array([P.conductances(MODELS[n])[:, k] for n in NAMES])
    print(f"g{cur:5s} bound {BOUNDS[cur]:7.0f}: deposited {v.min():8.2f} to {v.max():8.2f}; models with a cell above the bound "
          f"{int((v > BOUNDS[cur]).any(axis=1).sum())} of {len(NAMES)}")
tc = np.array([MODELS[n][[35, 72, 109]] for n in NAMES])
print(f"tau_Ca  bound 2000 ms: as stored {tc.min():.1f} to {tc.max():.1f} (above: {int((tc > 2000).any(axis=1).sum())} models); "
      f"as the Java uses it (x 10) {10 * tc.min():.0f} to {10 * tc.max():.0f} ms (above: {int((10 * tc > 2000).any(axis=1).sum())} models)")
gs = np.array([MODELS[n][111:118] for n in NAMES])
print(f"g syn   bound 0.05: deposited {gs.min():.4f} to {gs.max():.4f}; models with a synapse above the bound {int((gs > 0.05).any(axis=1).sum())} of {len(NAMES)}")

## deposit-models

> The deposit holds the code and the parameters of the paper's models, under the names the
> Methods give, and simulating them at 10 degrees C gives the control rhythm the models were
> fitted to, about 1 Hz with a PD duty cycle near 20% and LP and PY duty cycles near 25%.

Each model simulated 20 s at 10 degrees C from the deposited initial conditions; the first 10 s
dropped (criterion). Burst frequency and duty cycle are per-cycle means (Methods, Finding
parameters, "the average burst frequency" and "the average duty cycle").

In [ ]:
r10 = P.pmap(P.run_fixed, [(n, MODELS[n], CI, 10.0, 20.0, 10.0) for n in NAMES])
rows = []
for r in r10:
    row = {"model": r["key"]}
    for c, cell in enumerate(P.CELLS):
        b = P.burst_stats(r["spikes"][c])
        row[f"f {cell}"] = np.mean(b["freq"]) if len(b["freq"]) else np.nan
        row[f"dc {cell}"] = np.mean(b["dc"]) if len(b["dc"]) else np.nan
    row["NET"] = r["classes"][3]
    rows.append(row)
d10 = pd.DataFrame(rows).set_index("model")
ok_f = d10[[f"f {c}" for c in P.CELLS]].sub(1.0).abs().le(0.1).all(axis=1)
ok_pd = d10["dc PD"].sub(0.2).abs().le(0.02)
ok_lppy = d10[["dc LP", "dc PY"]].sub(0.25).abs().le(0.025).all(axis=1)
ok_lppy20 = d10[["dc LP", "dc PY"]].sub(0.2).abs().le(0.02).all(axis=1)
d10["f ok"], d10["PD dc ok"], d10["LP PY dc ok (0.25)"], d10["LP PY dc ok (0.20)"] = ok_f, ok_pd, ok_lppy, ok_lppy20
print(d10.round(3).to_string())
print("\nranges at 10 C:", {c: (round(d10[c].min(), 3), round(d10[c].max(), 3)) for c in d10.columns[:6]})
n_all = int((ok_f & ok_pd & ok_lppy).sum())
n_all20 = int((ok_f & ok_pd & ok_lppy20).sum())
print(f"\nwithin tolerance on all values (LP, PY target 0.25): {n_all} of {len(d10)}")
print(f"within tolerance on all values (LP, PY target 0.20, variant): {n_all20} of {len(d10)}")
print("within tolerance, frequency alone:", int(ok_f.sum()), "; PD duty cycle alone:", int(ok_pd.sum()),
      "; LP and PY duty cycle (0.25) alone:", int(ok_lppy.sum()), "; (0.20):", int(ok_lppy20.sum()))
print("triphasic at 10 C:", int((d10.NET == "triphasic").sum()), "of", len(d10))
print("mean burst frequency over models and cells: %.3f Hz (median %.3f)" %
      (d10[[f"f {c}" for c in P.CELLS]].values.mean(), np.median(d10[[f"f {c}" for c in P.CELLS]].values)))
print("mean PD duty cycle over models: %.3f; mean LP and PY duty cycle over models and the two cells: %.3f" %
      (d10["dc PD"].mean(), d10[["dc LP", "dc PY"]].values.mean()))
qg = np.array([P.q10_conductances(MODELS[n]) for n in NAMES]); qt = np.array([P.q10_timescales(MODELS[n]) for n in NAMES])
qs = np.array([P.q10_synapses(MODELS[n]) for n in NAMES])
print("Q10 of conductances: %.4f to %.4f (intrinsic), %.4f to %.4f (synaptic)" % (qg.min(), qg.max(), qs[:, :2].min(), qs[:, :2].max()))
print("Q10 of time scales: %.4f to %.4f (intrinsic), %.4f to %.4f (synaptic)" % (qt.min(), qt.max(), qs[:, 2:].min(), qs[:, 2:].max()))
print("all within [1, 2] and [1, 4]:", bool(qg.min() >= 1 and qg.max() <= 2 and qs[:, :2].max() <= 2 and qs.min() >= 1
                                             and qt.min() >= 1 and qt.max() <= 4 and qs[:, 2:].max() <= 4))

In [ ]:
# Burst frequency at 10 C, 20 s simulated and the first 10 s dropped: the Methods' target is 1 Hz.
print(d10[[f"f {c}" for c in P.CELLS]].describe().round(3).to_string())

**Result: not reproduced.** The deposit holds the model code and 38 parameter sets, not 36. Of
the 21 model names in the Methods, 17 are found (13 as printed, 2 with the letter O read as the
digit 0, 1 at the unreadable character, 1 with two characters swapped) and 4 are absent. Every
Q10 is within [1, 2] for conductances and [1, 4] for time scales. All 38 networks are triphasic
at 10 C, but only 3 of 38 models are within tolerance on every value (frequency 1 +- 0.1 Hz alone:
10 of 38; mean frequency over models and cells 1.246 Hz), and 0 of 38 with the 0.20 LP and PY
target of the variant. (conclusion written after the run)

## Simulations at 16 temperatures, 10 to 25 degrees C

Every model at 10, 11, ..., 25 degrees C (Figure 1 legend: "16 values of temperature between
10 C - 25 C", "average values over 30 s"). Each run is 40 s from the initial conditions; the
first 10 s are dropped (Methods, Finding parameters) and the last 30 s analysed.

In [ ]:
TEMPS = np.arange(10, 26).astype(float)
t0 = time.time()
scan = P.pmap(P.run_fixed, [(n, MODELS[n], CI, T, 40.0, 10.0) for n in NAMES for T in TEMPS])
SCAN = {(r["key"], r["temp"]): r for r in scan}
print(f"{len(scan)} runs in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

def summary(r):
    out = {}
    for c, cell in enumerate(P.CELLS):
        b = P.burst_stats(r["spikes"][c])
        out[f"f {cell}"] = np.mean(b["freq"]) if len(b["freq"]) else np.nan
        out[f"dc {cell}"] = np.mean(b["dc"]) if len(b["dc"]) else np.nan
    ph = P.phases(r["spikes"])
    for k, v in ph.items():
        out[k] = np.nanmean(v) if np.isfinite(v).any() else np.nan
    out["NET"] = r["classes"][3]
    return out

def table(name):
    return pd.DataFrame([summary(SCAN[(name, T)]) for T in TEMPS], index=TEMPS.astype(int))

def q10_readings(f):
    lf = np.log(f.values)
    a = np.exp(10 * np.polyfit(TEMPS, lf, 1)[0])
    b = (f.loc[25] / f.loc[10]) ** (10 / 15)
    c = f.loc[20] / f.loc[10]
    return a, b, c

PH_COLS = ["dc PD", "dc LP", "dc PY", "PD off", "LP on", "LP off", "PY on", "PY off"]

def constancy(t):
    """Readings (i) range within 0.1 of a cycle, (ii) every value within 20% of its 10 C value."""
    rng = t[PH_COLS].max() - t[PH_COLS].min()
    rel = (t[PH_COLS] / t.loc[10, PH_COLS] - 1).abs().max()
    return rng, rel

def report(name):
    t = table(name)
    print(f"model {name}")
    print(t.round(3).to_string())
    f = t[["f PD", "f LP", "f PY"]].mean(axis=1)
    qa, qb, qc = q10_readings(f)
    spread = (t[["f PD", "f LP", "f PY"]].max(axis=1) / t[["f PD", "f LP", "f PY"]].min(axis=1) - 1).max()
    rng, rel = constancy(t)
    print(f"Q10 of burst frequency (mean of the three cells): (a) fit {qa:.3f}, (b) (f25/f10)^(10/15) {qb:.3f}, (c) f20/f10 {qc:.3f}")
    for c in P.CELLS:
        print(f"   {c}: (a) {q10_readings(t['f ' + c])[0]:.3f}, (b) {q10_readings(t['f ' + c])[1]:.3f}, (c) {q10_readings(t['f ' + c])[2]:.3f}")
    print(f"largest spread between the cells' frequencies at one temperature: {100 * spread:.1f}%")
    print("range over temperatures (cycles):", rng.round(3).to_dict())
    print("largest relative change from 10 C:", rel.round(3).to_dict())
    print(f"(i) all within 0.1 of a cycle: {bool((rng <= 0.1).all())}; (ii) all within 20% of 10 C: {bool((rel <= 0.2).all())}")
    rises = {c: bool(t.loc[25, 'f ' + c] > t.loc[10, 'f ' + c]) for c in P.CELLS}
    print("frequency higher at 25 than at 10 C:", rises, "; network triphasic at", int((t.NET == "triphasic").sum()), "of 16")
    return t, (qa, qb, qc), rng, rel, rises

## fig1c-frequency-q10-phase-constant

> In the model of Figure 1, as temperature rises from 10 to 25 degrees C the burst frequency of
> all three cells increases with a Q10 near 2, while each cell's duty cycle and the phases of
> burst onsets and offsets stay approximately constant.

Model 4?6W66 of the Methods is 4X6W66 in the deposit (the only match). Phases are measured from
each PD burst start in units of that PD cycle (Figure 1 legend).

In [ ]:
t1, q1, rng1, rel1, rises1 = report(M["4?6W66"])
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for c in P.CELLS:
    ax[0].plot(t1.index, t1["f " + c], "o-", label=c); ax[1].plot(t1.index, t1["dc " + c], "o-", label=c)
for k in ["PD off", "LP on", "LP off", "PY on", "PY off"]:
    ax[2].plot(t1.index, t1[k], "o-", label=k)
ax[0].set_ylabel("burst frequency (Hz)"); ax[1].set_ylabel("duty cycle"); ax[2].set_ylabel("phase")
for a in ax: a.set_xlabel("temperature (C)"); a.legend(fontsize=7)
plt.tight_layout(); plt.show()
print("all plotted values are within the axes (autoscaled).")

**Result: partially reproduced.** In 4X6W66 the Q10 of burst frequency is 1.927 (fit), 1.982
(end points) and 1.937 (f20 / f10), within 2 +- 0.2 under every reading, and the three cells'
frequencies differ by at most 0.8%. Duty cycles and phases are not constant under either reading:
the LP duty cycle spans 0.11 of a cycle (reading i, bound 0.1) and changes by up to 34% from its
10 C value (reading ii, bound 20%); every other duty cycle and phase stays within 0.1 of a cycle.

## fig1s1-three-models

> In three further models with different maximal conductances and Q10 values, the duty cycles
> and phases also stay approximately constant while the frequency increases from 10 to 25
> degrees C.

In [ ]:
s1 = {}
for n in ["WWZ3CN", "FGFKQS", "71G6LA"]:
    s1[n] = report(M[n])
    print()
for n, (t, q, rng, rel, rises) in s1.items():
    print(f"{n}: frequency rises in every cell {all(rises.values())}; constant (i) {bool((rng <= 0.1).all())}, "
          f"(ii) {bool((rel <= 0.2).all())}; Q10 (a) {q[0]:.2f}")
print("FGFKQS shares its maximal conductances with 4X6W66 (see the data check):", key["FGFKQS"] == key["4X6W66"])

**Result: not reproduced.** In WWZ3CN, FGFKQS and 71G6LA the frequency rises from 10 to 25 C in
every cell (Q10 by the fit 2.05, 1.99 and 1.71), but none meets either reading of constancy: the
largest ranges are 0.171 (WWZ3CN, LP offset), 0.121 (FGFKQS, LP duty cycle) and 0.102 (71G6LA, PY
offset) of a cycle, and the largest changes from 10 C are 34%, 43% and 24%. FGFKQS has the same
maximal conductances as 4X6W66, with other Q10 values.

## all-models-compensated

> All 36 models produce a triphasic pyloric rhythm over the working range of 10 to 25 degrees
> C; for every set of maximal conductances tried, multiple sets of Q10 values that make the
> rhythm temperature robust could be found.

The deposit holds 38 models; all are tested, and the counts are given over the 38.

In [ ]:
fails = pd.DataFrame({n: [SCAN[(n, T)]["classes"][3] != "triphasic" for T in TEMPS] for n in NAMES}, index=TEMPS.astype(int))
per_model = fails.sum()
print("temperatures (of 16) at which the network is not triphasic, per model:")
print(per_model.to_string())
print(f"\nmodels triphasic at all 16 temperatures: {int((per_model == 0).sum())} of {len(NAMES)}")
print(f"models failing at 1 or more temperatures: {per_model[per_model > 0].to_dict()}")
for n in per_model[per_model > 0].index:
    bad = fails.index[fails[n]].tolist()
    print(f"  {n}: not triphasic at {bad}; cell classes there:", [SCAN[(n, float(T))]['classes'] for T in bad][:4])
strict = pd.DataFrame({n: [not P.triphasic(SCAN[(n, T)]["spikes"], strict=True) for T in TEMPS] for n in NAMES}, index=TEMPS.astype(int))
print(f"\nvariant, order required in every cycle: models triphasic at all 16 temperatures {int((strict.sum() == 0).sum())} of {len(NAMES)}")
print(f"\nsets of maximal conductances with more than one deposited Q10 set: {len(multi)} of {len(groups)}")
for g in multi:
    print(f"  {g}: compensated (triphasic at all 16) {[bool(per_model[n] == 0) for n in g]}")

**Result: partially reproduced.** 36 of the 38 deposited models are triphasic at all 16
temperatures from 10 to 25 C; ED50XO (not triphasic at 19 and 21 C, all cells irregular bursting)
and MAXTTP (10 and 14 C, LP irregular bursting) are not. With the order required in every cycle, 35
of 38 are. The deposit holds more than one Q10 set for 9 of its 27 distinct sets of maximal
conductances, and every one of those 20 models is compensated, so multiple Q10 sets are found for
some, not all, sets of conductances in the deposit.

## figs3-duty-cycle-variable-across-models

> Across the 36 models the LP duty cycle depends on temperature in different ways, so that the
> duty cycle distributions differ from model to model.

The criterion's main reading, the 60 min ramp of every model, is not run; `ramps.ipynb` prints
its estimate from a subset, which would need a notebook of its own. This is its variant: per-burst LP duty cycles from the 16
fixed temperatures (30 s each), pooled per model; two-sample Kolmogorov-Smirnov test for every
pair of the 38 deposited models (703 pairs).

In [ ]:
DC = {n: np.concatenate([P.burst_stats(SCAN[(n, T)]["spikes"][1])["dc"] for T in TEMPS]) for n in NAMES}
summ = pd.DataFrame({n: dict(bursts=len(d), median=np.median(d), lo=np.min(d), hi=np.max(d)) for n, d in DC.items()}).T
print(summ.round(3).to_string())
pv = np.array([stats.ks_2samp(DC[a], DC[b]).pvalue for a, b in combinations(NAMES, 2)])
print(f"\npairs: {len(pv)}; with p < 0.05: {(pv < 0.05).sum()} ({(pv < 0.05).mean():.3f})")
for g in multi:
    for a, b in combinations(g, 2):
        print(f"   same conductances, different Q10: {a} vs {b}: KS p = {stats.ks_2samp(DC[a], DC[b]).pvalue:.2e}")
lp = pd.DataFrame({n: [np.mean(P.burst_stats(SCAN[(n, T)]["spikes"][1])["dc"]) for T in TEMPS] for n in NAMES}, index=TEMPS.astype(int))
rho = lp.apply(lambda c: stats.spearmanr(lp.index, c)[0])
print("Spearman correlation of mean LP duty cycle with temperature, per model: from %.2f to %.2f; "
      "below -0.5: %d, above 0.5: %d, between: %d" % (rho.min(), rho.max(), (rho < -0.5).sum(), (rho > 0.5).sum(),
                                                      ((rho >= -0.5) & (rho <= 0.5)).sum()))

**Result: reproduced** (on the criterion's variant reading, 16 fixed temperatures; the ramp
reading was not run). All 703 pairs of the 38 models have LP duty-cycle distributions that differ
at p < 0.05 (two-sample Kolmogorov-Smirnov), including every pair that shares its maximal
conductances (largest p 0.025). The test pools autocorrelated per-burst values across
temperatures, so it detects small differences, and the outcome carries little more than that the
distributions are not identical. Model medians and ranges are in the table; the mean LP duty cycle
falls with temperature in 23 models (Spearman below -0.5), rises in 3 and neither in 12.

## intermediate-temperature-failures

> Of the Q10 solutions that met the target at the control temperatures, 2 in 40 did not display
> the correct activity at temperatures in between.

Blocked (data_missing): the deposit holds 38 parameter sets with no record of which solutions
were screened or discarded. The 38 are not the 40 screened solutions as far as the files show;
the count of models above is all that can be said. The cell below prints what the deposit says
about its models: the README's lines that mention models or solutions, and the score in each file
name against the models that fail between 10 and 25 degrees C.

In [ ]:
readme = Z.read(P.ROOT + "README.txt").decode()
print("README lines that mention models, solutions, scores or counts:")
for line in readme.splitlines():
    if re.search(r"model|solution|score|discard|36|38|40", line, re.I):
        print("   ", line.strip()[:160])
sc = pd.Series({n: P.score_of(FILES[n]) for n in NAMES}).sort_values()
print("\nscore in the file name, per model (ascending):")
print(sc.round(1).to_string())
print("\nmodels not triphasic at some temperature from 10 to 25 C:", {n: round(sc[n], 1) for n in per_model[per_model > 0].index})
print("median score %.1f; the three largest: %s" % (sc.median(), sc.round(1).tail(3).to_dict()))

## fig3-current-amplitudes

> In the PD cell of one model, from 10 to 25 degrees C the Na current peak more than doubles,
> the CaT current peak falls to about half despite its larger conductance, and the KCa current
> grows about six-fold; in a second model the Na current falls slightly and the KCa current
> falls moderately.

PD cell, 13 s simulated, the first 10 s dropped and 3 s analysed, as the deposit's currentscape
script does (integratePyloricSQ10s_currentscapes_RK4_mp.py: transient = 10, nsecs = 3, dt = 0.05).
Readings of the peak: (a) the largest absolute value over the 3 s; (b) the mean over PD cycles
(PD burst start to the next) of each cycle's largest absolute value.

In [ ]:
def states(name, temp, seconds=13.0, record=3.0, p=None):
    return P.run_states((name, MODELS[name] if p is None else p, CI, temp, seconds, record))

ST = {}
for n in ["4X6W66", "SJR46Y", "WWZ3CN", "FGFKQS", "71G6LA", "J9U8SQ"]:
    for T in (10.0, 25.0):
        ST[(n, T)] = states(n, T)

def cell_currents(r, name, c):
    return P.currents_from_states(r["states"], MODELS[name], r["temp"])[c]

def peak_ratios(name):
    out = {}
    for rd in ("a", "b"):
        pk = {}
        for T in (10.0, 25.0):
            r = ST[(name, T)]
            cur = np.abs(cell_currents(r, name, 0))
            if rd == "a":
                pk[T] = cur.max(axis=1)
            else:
                st = P.burst_stats(r["spikes"][0])["starts"]
                idx = (st / r["dt"]).astype(int)
                pk[T] = np.mean([cur[:, i:j].max(axis=1) for i, j in zip(idx[:-1], idx[1:])], axis=0)
        out[rd] = pd.DataFrame({"peak 10 C": pk[10.0], "peak 25 C": pk[25.0], "ratio": pk[25.0] / pk[10.0]},
                               index=P.CURRENTS + ["Syn"])
    return out

In [ ]:
fig3 = {n: peak_ratios(n) for n in ["4X6W66", "SJR46Y"]}
for n, d in fig3.items():
    for rd, t in d.items():
        print(f"{n}, PD, reading ({rd}):"); print(t.round(3).to_string()); print()

def fig3_checks(a, b, rd):
    A, B = fig3[a][rd]["ratio"], fig3[b][rd]["ratio"]
    return {"A Na > 2": A["Na"] > 2, "A CaT 0.5 +- 0.05": abs(A["CaT"] - 0.5) <= 0.05, "A KCa 6 +- 0.6": abs(A["KCa"] - 6) <= 0.6,
            "B Na < 1": B["Na"] < 1, "B KCa < 1": B["KCa"] < 1}

for a, b in [("4X6W66", "SJR46Y"), ("SJR46Y", "4X6W66")]:
    for rd in ("a", "b"):
        ch = fig3_checks(a, b, rd)
        print(f"A = {a}, B = {b}, reading ({rd}): {sum(ch.values())} of 5 hold: {ch}")

**Result: partially reproduced.** In 4X6W66 (panel A as listed) the PD Na peak grows 2.42-fold
(more than 2), the CaT peak falls to 0.486 of its 10 C value (0.5 +- 0.05), and the KCa peak grows
8.9-fold (reading a) or 9.1-fold (reading b), outside 6 +- 0.6. In SJR46Y the Na peak falls to
0.81 and the KCa peak to 0.80. Four of the five parts hold under both readings; with the panels
swapped, none does.

## fig4-currentscapes-reorganize

> In the model of Figure 4, currentscapes differ between 10 and 25 degrees C; in PD the leak
> current's inward share is seen at both the start and end of the burst at 10 degrees C but
> mainly at the start at 25 degrees C, the A current's share is even across the burst at 10
> degrees C and larger at its start at 25 degrees C, and in all cells, most in LP and PY,
> temperature raises the KCa share toward the end of bursts.

WWZ3CN, the same 3 s windows. Where a burst starts and ends is open. Two bounds are run: (spikes)
from its first to its last spike crossing (bursts grouped with the 200 ms threshold), and (slow
wave) from the last upward crossing of the Methods' -50 mV slow-wave threshold before its first
spike to the first downward crossing after its last spike. Every leak reversal potential is
-50 mV, so the leak current is inward only below -50 mV, which the spike bounds exclude. Shares are averaged over the samples of each part of each
burst, then over bursts. The A current's share is its outward share (IA is a potassium current).

In [ ]:
def burst_part_shares(r, name, c, frac, bounds="spikes"):
    cur = cell_currents(r, name, c)
    out_sh, in_sh = P.shares(cur)
    if bounds == "spikes":
        b = [(int(s / r["dt"]), int(e / r["dt"]), k) for s, e, k in zip(*P.bursts(r["spikes"][c]))]
    else:
        b = P.slow_wave_bursts(r["states"][:, 13 * c], r["spikes"][c], r["dt"])
    res = {"in start": [], "in end": [], "out start": [], "out end": []}
    for i, j, k in b:
        if k < 2:
            continue
        w = max(1, int((j - i) * frac))
        res["in start"].append(in_sh[:, i:i + w].mean(axis=1)); res["in end"].append(in_sh[:, j - w:j].mean(axis=1))
        res["out start"].append(out_sh[:, i:i + w].mean(axis=1)); res["out end"].append(out_sh[:, j - w:j].mean(axis=1))
    return {k: pd.Series(np.mean(v, axis=0), index=P.CURRENTS + ["Syn"]) for k, v in res.items()}, len(res["in start"])

fig4 = {}
for bounds in ("spikes", "slow wave"):
    for frac, label in [(1 / 3, "a, thirds"), (0.2, "b, 20%")]:
        sh = {(c, T): burst_part_shares(ST[("WWZ3CN", T)], "WWZ3CN", c, frac, bounds) for c in range(3) for T in (10.0, 25.0)}
        pd10, pd25 = sh[(0, 10.0)][0], sh[(0, 25.0)][0]
        with np.errstate(invalid="ignore", divide="ignore"):
            leak10 = pd10["in end"]["Leak"] / pd10["in start"]["Leak"]
            leak25 = pd25["in end"]["Leak"] / pd25["in start"]["Leak"]
        a10 = pd10["out start"]["A"] / pd10["out end"]["A"]
        a25 = pd25["out start"]["A"] / pd25["out end"]["A"]
        kca = {P.CELLS[c]: (sh[(c, 10.0)][0]["out end"]["KCa"], sh[(c, 25.0)][0]["out end"]["KCa"]) for c in range(3)}
        inc = {k: v[1] - v[0] for k, v in kca.items()}
        print(f"bounds ({bounds}), reading ({label}); bursts used per cell and temperature:", {f"{P.CELLS[c]} {int(T)}": sh[(c, T)][1] for c, T in sh})
        print(f"  PD leak inward share, end over start: 10 C {leak10:.3f} (start {pd10['in start']['Leak']:.4f}, end {pd10['in end']['Leak']:.4f}); "
              f"25 C {leak25:.3f} (start {pd25['in start']['Leak']:.4f}, end {pd25['in end']['Leak']:.4f})")
        print(f"  PD A outward share, start over end: 10 C {a10:.3f}; 25 C {a25:.3f}")
        print("  KCa outward share at the end of bursts (10 C, 25 C):", {k: tuple(round(x, 4) for x in v) for k, v in kca.items()})
        print("  increase:", {k: round(v, 4) for k, v in inc.items()})
        parts = {"leak": bool(leak25 < leak10), "A": bool((0.8 <= a10 <= 1.25) and a25 > a10),
                 "KCa": bool(all(v > 0 for v in inc.values()) and inc["LP"] > inc["PD"] and inc["PY"] > inc["PD"])}
        print("  parts that hold:", parts)
        fig4[(bounds, label)] = parts

**Result: not reproduced.** In WWZ3CN the PD leak current has no inward share inside the bursts
at either temperature under either burst bound (0.0000 at start and end): every leak reversal
potential is -50 mV, so the leak is outward while V is above -50 mV, which holds between the
first and last spike and also between the slow-wave crossings. The leak part cannot hold under
any burst bound that keeps V above -50 mV. The PD A share at the start over the end of the burst is
1.278 at 10 C with spike bounds and 1.364 or 1.370 with slow-wave bounds, outside 0.8 to 1.25 in
every case. With spike bounds the KCa share at the end of bursts rises from 10 to 25 C in LP
(+0.29, +0.34) and PY (+0.22, +0.25) but falls in PD (-0.21, -0.20); with slow-wave bounds it rises
in all three cells but least in LP (+0.03, against PD +0.34 and +0.62), so the KCa part fails under
all four readings.

## currentscapes-dominant-currents

> In all models and at all temperatures, spikes are driven first by INa and then by IKd, which
> then dominate the inward and outward currents, and when the cells are most hyperpolarized IH
> and the synaptic currents dominate.

Every model, every cell, at 10, 15, 20 and 25 degrees C; 13 s simulated and the last 3 s used.

In [ ]:
t0 = time.time()
dom = P.pmap(P.dominant_summary, [(n, MODELS[n], CI, T, 13.0, 3.0) for n in NAMES for T in (10.0, 15.0, 20.0, 25.0)])
print(f"{len(dom)} runs in {time.time() - t0:.0f} s")
rows = [dict(model=r["key"], temp=r["temp"], **c) for r in dom for c in r["cells"]]
dd = pd.DataFrame(rows)
nsp = dd.spikes.sum()
print(f"spikes: {nsp}; Na largest inward share in the 1 ms before the peak: {dd.na_first.sum() / nsp:.3f}; "
      f"Kd largest outward share in the 1 ms after it: {dd.kd_after.sum() / nsp:.3f}")
names9 = P.CURRENTS + ["Syn"]
dd["low_in_name"] = [names9[i] for i in dd.low_in]; dd["low_out_name"] = [names9[i] for i in dd.low_out]
print("largest inward share at the lowest 5% of V:", dd.low_in_name.value_counts().to_dict())
print("largest outward share at the lowest 5% of V:", dd.low_out_name.value_counts().to_dict())
main = (dd.low_in == 6) & (dd.low_out == 8)
var = dd.low_top2.apply(lambda x: x == [6, 8])
print(f"cases (model, cell, temperature) with H inward and Syn outward largest: {main.mean():.3f} of {len(dd)}")
print(f"variant, H and Syn the two largest shares on either side: {var.mean():.3f}")
print("per cell:", dd.assign(main=main).groupby("cell").main.mean().round(3).to_dict())
print("per temperature:", dd.assign(main=main).groupby("temp").main.mean().round(3).to_dict())
print("per cell, Na and Kd fractions:", dd.groupby("cell")[["spikes", "na_first", "kd_after"]].sum().assign(
    na=lambda x: x.na_first / x.spikes, kd=lambda x: x.kd_after / x.spikes).round(3).to_dict("index"))

**Result: reproduced.** Over 38 models, 3 cells and 4 temperatures (26,868 spikes), Na holds the
largest inward share in the 1 ms before every spike peak (fraction 1.000) and Kd the largest
outward share in the 1 ms after it (1.000). At the 5% of time with the lowest voltage, H holds the
largest inward share in 456 of 456 cases and the synaptic current the largest outward share in
453 (KCa in 3), so both hold in 0.993 of cases, above the 0.9 bound.

## fig5-a-to-kca-tradeoff

> In the 20 ms after the PD burst, the share of the A current falls smoothly and the share of
> the KCa current rises smoothly as temperature goes from 10 to 25 degrees C ...

FGFKQS (Methods) and J9U8SQ (legend, "same model as in Figure 7A"). PD, the 20 ms after the
last spike of each burst (from its -20 mV crossing), outward shares averaged over samples and
bursts, at the 16 temperatures; 13 s simulated, the last 3 s used.

In [ ]:
res5 = P.pmap(P.end_shares, [(n, MODELS[n], CI, T, 13.0, 3.0) for n in ["FGFKQS", "J9U8SQ"] for T in TEMPS])
fig5 = {}
for n in ["FGFKQS", "J9U8SQ"]:
    t = pd.DataFrame([dict(zip(names9, s), T=T, bursts=b) for m, T, s, b in res5 if m == n]).set_index("T")
    rA = stats.spearmanr(t.index, t["A"])[0]; rK = stats.spearmanr(t.index, t["KCa"])[0]
    cross = t.loc[10.0, "A"] > t.loc[10.0, "KCa"] and t.loc[25.0, "KCa"] > t.loc[25.0, "A"]
    fig5[n] = dict(rA=rA, rK=rK, cross=cross)
    print(n); print(t.round(3).to_string())
    print(f"  Spearman with temperature: A {rA:.3f}, KCa {rK:.3f}; A > KCa at 10 C and KCa > A at 25 C: {cross}\n")

**Result: reproduced.** In FGFKQS (the model the Methods name), over the 20 ms after the PD
burst the A share falls from 0.312 to 0.082 and the KCa share rises from 0.077 to 0.577 from 10 to
25 C (Spearman -1.000 and 1.000), with A above KCa at 10 C and KCa above A at 25 C. In J9U8SQ (the
legend's reading) both trends hold (-0.997, 0.953) but KCa exceeds A already at 10 C.

## Simulations above 25 degrees C

Every model at 26 to 35 degrees C, 40 s, the first 10 s dropped, as for the working range.
These runs serve fig6s1-crash-mechanisms (the crash temperature of each model) and the counts
reported for fig7-high-temperature-slow-timescales.

In [ ]:
HOT = np.arange(26, 36).astype(float)
t0 = time.time()
hot = P.pmap(P.run_fixed, [(n, MODELS[n], CI, T, 40.0, 10.0) for n in NAMES for T in HOT])
for r in hot:
    SCAN[(r["key"], r["temp"])] = r
print(f"{len(hot)} runs in {time.time() - t0:.0f} s")
failed = sorted((r["key"], r["temp"]) for r in SCAN.values() if r["classes"][0] == "numerical failure")
print(f"runs whose integration diverged (NaN), of {len(NAMES) * (len(TEMPS) + len(HOT))}:", len(failed), failed)
crashT = {n: next((T for T in HOT if SCAN[(n, T)]["classes"][3] != "triphasic" and SCAN[(n, T)]["classes"][0] != "numerical failure"), None)
          for n in NAMES}
print("lowest temperature above 25 C at which the network is not triphasic (diverged runs skipped):", crashT)

In [ ]:
# The deposited integrator (RK4, dt = 0.05 ms, from the deposited initial conditions) returns NaN for some
# models at high temperature. The port gives NaN where the Java does: G7P2WE at 35 C, LP first, after about 0.1 s.
s35 = P.simulate(MODELS["G7P2WE"], 35.0, 0.2, CI, rec_every=1)
bad = np.where(~np.isfinite(s35.rec).all(axis=1))[0]
print("G7P2WE, 35 C: first sample with NaN", bad[0] if len(bad) else None, "of", len(s35.rec), "(step 0.05 ms);",
      "V there:", s35.rec[bad[0] - 1] if len(bad) else None)
print("models with a diverged run at 26 to 35 C:", sorted({n for n, T in failed}))

## fig6-crash-different-mechanisms

> Two models that are phase compensated at 10 and 25 degrees C crash at 30 degrees C with
> complete loss of PD activity, by different currents in each.

FGFKQS and 71G6LA at 10, 25 and 30 degrees C. Classes from the 40 s runs (last 30 s); the
dominant currents, the largest mean inward and outward share in PD, over the last 3 s of the same
40 s run (the run is deterministic, so it is repeated with the states recorded). The last 3 s of
a 13 s run, as the deposit's currentscape script takes them, are shown as a variant.

In [ ]:
def dominant_pair(r, name, c):
    cur = P.currents_from_states(r["states"], MODELS[name], r["temp"])[c]
    out_sh, in_sh = P.shares(cur)
    return names9[int(np.argmax(in_sh.mean(axis=1)))], names9[int(np.argmax(out_sh.mean(axis=1)))]

fig6 = {}
for n in ["FGFKQS", "71G6LA"]:
    r30 = states(n, 30.0, seconds=40.0)
    cl = {T: SCAN[(n, T)]["classes"] for T in (10.0, 25.0, 30.0)}
    pair = dominant_pair(r30, n, 0)
    pair13 = dominant_pair(states(n, 30.0), n, 0)
    nspk = len(SCAN[(n, 30.0)]["spikes"][0])
    fig6[n] = dict(classes=cl, pair=pair, pd_spikes_30=nspk)
    print(n, cl, f"; PD spikes in the 30 s analysed at 30 C: {nspk}; PD dominant inward, outward at 30 C: {pair} "
          f"(variant, last 3 s of a 13 s run: {pair13})")
    print("   PD V range at 30 C (last 3 s): %.1f to %.1f mV" % (r30["states"][:, 0].min(), r30["states"][:, 0].max()))
same_pair = fig6["FGFKQS"]["pair"] == fig6["71G6LA"]["pair"]
print("dominant pairs the same in the two models:", same_pair)

**Result: not reproduced.** FGFKQS and 71G6LA are triphasic at 10 and 25 C. At 30 C, 71G6LA's PD
is quiescent (no spike in 30 s), but FGFKQS's PD fires 1005 spikes in 30 s (class other), so the
complete loss of PD activity holds in one model of two. The dominant PD currents at 30 C, from the
last 3 s of the same 40 s runs, differ (H and KCa in FGFKQS, H and Syn in 71G6LA), as they do in
the 13 s variant.

## fig6s1-crash-mechanisms

> In eight further models, currentscapes at crashed states show that the networks crash by
> different mechanisms in each case.

The Methods name B7SDOL, J9U8SQ, 4R2719, R3VIEW, Q8FBN1, RMZ8KZ, EF84RN and SJR46Y. Q8FBN1 is not
in the deposit; B7SDOL is B7SD0L. The seven are tested. Each is taken at its crash temperature
from the 40 s runs, and the dominant currents are taken over the last 3 s of the same 40 s run;
the last 3 s of a 13 s run are a variant.

In [ ]:
s6 = [M[n] for n in METHODS_NAMES["Figure 6-S1"] if isinstance(M[n], str)]
print("models tested:", s6)
pat, pat13 = {}, {}
for n in s6:
    T = crashT[n]
    if T is None:
        print(f"{n}: triphasic at every temperature up to 35 C"); continue
    r = states(n, T, seconds=40.0)
    pat[n] = tuple(dominant_pair(r, n, c) for c in range(3))
    pat13[n] = tuple(dominant_pair(states(n, T), n, c) for c in range(3))
    print(f"{n}: crashes at {T:.0f} C, classes {SCAN[(n, T)]['classes']}; dominant (inward, outward) per cell {pat[n]}; "
          f"variant (13 s run) {pat13[n]}")
distinct = len(set(pat.values()))
print(f"\nmodels that crash below 35 C: {sum(crashT[n] is not None and crashT[n] < 35 for n in s6)} of {len(s6)}; "
      f"distinct patterns: {distinct} (variant, 13 s runs: {len(set(pat13.values()))})")
print("models sharing a pattern:", [(a, b) for a, b in combinations(pat, 2) if pat[a] == pat[b]])
print("variant, models sharing a pattern:", [(a, b) for a, b in combinations(pat13, 2) if pat13[a] == pat13[b]])
# What the missing Q8FBN1 could change: it adds at most one pattern, and whether it crashes cannot be shown.
best = distinct + (8 - len(s6))
print(f"with Q8FBN1 (absent): at most {best} distinct patterns among 8; reproduced (8 distinct) possible: {best == 8}; "
      f"partially reproduced (6 or more distinct, every model crashing) possible: {best >= 6}")

**Result: blocked (data_missing).** Seven of the eight models are in the deposit (Q8FBN1 is absent).
All seven stop being triphasic below 35 C (at 26 to 32 C), but their dominant (inward, outward)
currents, from the last 3 s of the same 40 s runs, form 5 distinct patterns: B7SD0L, J9U8SQ and
SJR46Y share H and Syn in all three cells. The 13 s variant also gives 5, with EF84RN in place of
SJR46Y. The criterion asks that no two share a pattern (reproduced) or at least six distinct among
eight (partially). With the eighth model at most six could be distinct, as printed, so reproduced
is ruled out but partially reproduced is not: it would need Q8FBN1 to crash with a sixth pattern,
which the deposit cannot show. The outcome therefore depends on the missing model, as for
fig7-high-temperature-slow-timescales, so the claim is blocked (data_missing); it can be at best
partially reproduced.

## fig7-high-temperature-slow-timescales

> At temperatures above 25 degrees C the models crash in different ways (quiescence,
> subthreshold oscillations, or irregular activity), and in five models at 29 to 34 degrees C
> time scales much longer than the 1 s period at 10 degrees C emerge.

Blocked (data_missing): three of the five models of Figure 7 (G81PUL, G7EOZ8, SAXT8Y) are not in
the deposit. The two that are (J9U8SQ at 34 C, KNMLBC filed as KMNLBC at 30 C), and FGFKQS at
34 C (the variant for panel A), are shown for information.

Reading (a) is the longest interval without a spike in each cell; reading (b) the dominant period
of the slow voltage: V every 1 ms over the last 30 s of the 40 s run, spikes removed with a 50 ms
median filter (criterion), and the period of the largest peak of the periodogram (resolution
1/30 Hz).

In [ ]:
def longest_gaps(r):
    out = []
    for c in range(3):
        s = r["spikes"][c]
        ev = np.r_[0.0, s, r["length"]]
        out.append(np.max(np.diff(ev)))
    return out

from scipy import ndimage, signal

def slow_periods(name, temp):
    """Reading (b): dominant period (s) of each cell's median-filtered V over the last 30 s of a 40 s run."""
    s = P.simulate(MODELS[name], temp, 40.0, CI, rec_every=20, rec_from=10.0)
    if not s.finite:
        return None
    out = []
    for c in range(3):
        v = ndimage.median_filter(s.rec[:, c], size=51)
        f, pw = signal.periodogram(v - v.mean(), fs=1 / s.rec_dt)
        k = 1 + int(np.argmax(pw[1:]))
        out.append(1 / f[k])
    return out

for n, T in [("J9U8SQ", 34.0), ("KMNLBC", 30.0), ("FGFKQS", 34.0)]:
    r = SCAN[(n, T)]
    per = slow_periods(n, T)
    print(f"{n} at {T:.0f} C: classes {r['classes']}; (a) longest interval without a spike per cell (s, 30 s window, edges included): "
          f"{np.round(longest_gaps(r), 2)}; (b) dominant period of the slow voltage per cell (s): "
          f"{np.round(per, 2) if per is not None else 'not computed, the run diverged'}")
cnt = Counter()
for n in NAMES:
    for T in HOT:
        cl = SCAN[(n, T)]["classes"]
        if cl[3] != "triphasic":
            cnt[tuple(cl[:3])] += 1
print("\ncell classes (PD, LP, PY) of non-triphasic networks over all models at 26 to 35 C:")
for k, v in cnt.most_common(12):
    print(f"  {v:4d}  {k}")

## fig9a-remove-a-current

> Removing the A current at 10 degrees C makes the network irregular, while the same deletion at
> 25 degrees C leaves a nearly normal triphasic rhythm except for irregular PY bursting.

WWZ3CN. Timings: (code) the deposit's integratePyloricSQ10s_remove_gA_onesteps_twotemps_RK4.py,
10 s transient, 5 s control, then 10 s with IA removed from the final state; (Results) 20 s of
control, then 20 s with IA removed; (legend) removal at 4 s, the length after it taken as 20 s.
Classes over the period after removal, its first 2 s dropped (criterion).

In [ ]:
TIMINGS = {"code": (15.0, 10.0), "Results": (20.0, 20.0), "legend": (4.0, 20.0)}

def removal(name, current, temp, control, after):
    p = MODELS[name]
    s1 = P.simulate(p, temp, control, CI)
    s2 = P.simulate(P.scaled(p, current, 0.0), temp, after, s1.final)
    before = P.classify(s1, max(0.0, control - 10.0), control)
    return before, P.classify(s2, 2.0, after), s1, s2

def mean_freq(sim, t0, t1):
    f = []
    for s in sim.window(t0, t1):
        b = P.burst_stats(s)
        f.append(np.mean(b["freq"]) if len(b["freq"]) else np.nan)
    return np.round(f, 3)

In [ ]:
fig9a = {}
for tm, (ctl, aft) in TIMINGS.items():
    for T in (10.0, 25.0):
        b, a, s1, s2 = removal("WWZ3CN", 3, T, ctl, aft)
        fig9a[(tm, T)] = a
        print(f"{tm:8s} {T:.0f} C: before {b}; after removal {a}")
for tm in TIMINGS:
    a10, a25 = fig9a[(tm, 10.0)], fig9a[(tm, 25.0)]
    p10 = a10[3] != "triphasic" and any(x in ("irregular bursting", "irregular spiking") for x in a10[:3])
    p25 = a25[0] == "regular bursting" and a25[1] == "regular bursting" and a25[2] == "irregular bursting"
    p25b = a25[2] == "irregular bursting"
    print(f"{tm}: 10 C part {p10}; 25 C part {p25} (PY irregular {p25b})")

**Result: partially reproduced.** In WWZ3CN, removing IA at 10 C leaves the network not
triphasic with irregular bursting cells under all three timings. At 25 C the network is never
the near-normal triphasic rhythm with only PY irregular: under the Results' timing (20 s, then 20
s without IA) all three cells burst irregularly and the network is not triphasic, so PY is
irregular but other cells are not regular, the criterion's partial case.

## fig9b-remove-kca-current

> In another model, removing the KCa current at 10 degrees C slows the rhythm but keeps it
> triphasic, while the same deletion at 25 degrees C silences the network ...

FGFKQS (Methods; also the deposit's script, "Fig. 13B") and J9U8SQ (legend), the same timings.

In [ ]:
fig9b = {}
for n in ["FGFKQS", "J9U8SQ"]:
    for tm, (ctl, aft) in TIMINGS.items():
        b10, a10, s1, s2 = removal(n, 4, 10.0, ctl, aft)
        f_before, f_after = mean_freq(s1, max(0.0, ctl - 10.0), ctl), mean_freq(s2, 2.0, aft)
        b25, a25, _, _ = removal(n, 4, 25.0, ctl, aft)
        ok10 = b10[3] == "triphasic" and a10[3] == "triphasic" and np.nanmean(f_after) < np.nanmean(f_before)
        ok25 = all(x == "quiescent" for x in a25[:3])
        fig9b[(n, tm)] = ok10 and ok25
        print(f"{n}, {tm}: 10 C before {b10} f {f_before}, after {a10} f {f_after}; 25 C after {a25}; "
              f"10 C part {ok10}, 25 C part {ok25}")

**Result: reproduced.** In FGFKQS (the Methods' model and the deposit's script), removing KCa at
10 C keeps the network triphasic and slows it (PD 1.061 to 0.904 Hz with the code's timing, 1.067
to 0.918 Hz with the Results' timing), and at 25 C all three cells are quiescent after removal.
The legend's timing (removal at 4 s, from the initial conditions) and J9U8SQ do not show it.

## fig10a-graded-removal

> In the model of Figure 10A, reducing gNa to 75% makes LP burst irregularly and the network
> non-triphasic at 10 degrees C but not at 25 degrees C; ...

4X6W66, each of gNa, gKCa, gCaT and gCaS at 100, 75, 50, 25 and 0% in all three cells, at 10 and
25 degrees C; each perturbed run 20 s, the first 10 s dropped (criterion). When the conductance is
changed is open for Figure 10, so two protocols are run:

- (control first) 20 s with every conductance intact from the deposited initial conditions, then
  the perturbed run from the final state. This is the protocol the Results give for the removals
  of Figure 9, in the same section ("the models were simulated for 20 s in control conditions and
  then IA was completely removed for the next 20 s"), and the one of the deposit's removal script
  (integratePyloricSQ10s_remove_gA_onesteps_twotemps_RK4.py: a control run, then the conductance
  scaled and the run continued from its final state). It is the main protocol (an assumption).
- (from t = 0) the perturbed model from the deposited initial conditions, a variant.

The grids are also given with a 0 mV spike threshold (variant).

In [ ]:
FR = [1.0, 0.75, 0.5, 0.25, 0.0]
CUR10 = {"Na": 0, "KCa": 4, "CaT": 1, "CaS": 2}
# Control runs, 20 s from the initial conditions: the final states of deposit-models at 10 C, and new runs at 25 C.
ctl25 = P.pmap(P.run_fixed, [(n, MODELS[n], CI, 25.0, 20.0, 10.0) for n in NAMES])
CTRL = {(r["key"], 10.0): r["final"] for r in r10}
CTRL.update({(r["key"], 25.0): r["final"] for r in ctl25})

def grid10a(start, thr=P.SPIKE_THRESHOLD):
    y0 = (lambda T: CTRL[("4X6W66", T)]) if start == "control first" else (lambda T: CI)
    tasks = [((cur, fr, T), P.scaled(MODELS["4X6W66"], k, fr), y0(T), T, 20.0, 10.0, thr)
             for cur, k in CUR10.items() for fr in FR for T in (10.0, 25.0)]
    return {r["key"]: r["classes"] for r in P.pmap(P.run_fixed, tasks)}

def checks10a(g):
    return {
        "Na 75% 10 C: LP irregular bursting, NET not triphasic": g[("Na", 0.75, 10.0)][1] == "irregular bursting" and g[("Na", 0.75, 10.0)][3] != "triphasic",
        "Na 75% 25 C: NET triphasic": g[("Na", 0.75, 25.0)][3] == "triphasic",
        "KCa every step 10 C: NET triphasic": all(g[("KCa", fr, 10.0)][3] == "triphasic" for fr in FR),
        "KCa 0% 25 C: all quiescent": all(x == "quiescent" for x in g[("KCa", 0.0, 25.0)][:3]),
        "CaT and CaS 0%: not triphasic at both, classes differ": all(
            g[(c, 0.0, 10.0)][3] != "triphasic" and g[(c, 0.0, 25.0)][3] != "triphasic" and g[(c, 0.0, 10.0)][:3] != g[(c, 0.0, 25.0)][:3]
            for c in ("CaT", "CaS")),
    }

G10 = {}
for start in ("control first", "from t = 0"):
    for thr in (P.SPIKE_THRESHOLD, 0.0):
        G10[(start, thr)] = grid10a(start, thr)
for start in ("control first", "from t = 0"):
    g = G10[(start, P.SPIKE_THRESHOLD)]
    print(f"=== protocol ({start}), spike threshold -20 mV")
    for cur in CUR10:
        for T in (10.0, 25.0):
            print(f"{cur} at {T:.0f} C")
            print(pd.DataFrame([g[(cur, fr, T)] for fr in FR], index=[f"{int(100 * fr)}%" for fr in FR],
                               columns=["PD", "LP", "PY", "NET"]).to_string())
checks10 = {}
for key, g in G10.items():
    checks10[key] = checks10a(g)
    print(f"\nprotocol ({key[0]}), spike threshold {key[1]:.0f} mV:")
    for k, v in checks10[key].items():
        print(f"   {v!s:5s}  {k}")
    print("   matching:", sum(checks10[key].values()), "of 5")
g = G10[("control first", 0.0)]
print("\nKCa at 10 C, protocol (control first), 0 mV threshold:", {f"{int(100 * fr)}%": g[("KCa", fr, 10.0)] for fr in FR})

**Result: partially reproduced** (under both protocols). With the control period first (main
protocol), four of the five stated grid cells match: Na at 75% makes LP burst irregularly and the
network not triphasic at 10 C and leaves it triphasic at 25 C; KCa removal leaves the network
triphasic at every step at 10 C; KCa at 0% at 25 C makes every cell quiescent. CaT and CaS at 0%
fail: the network is not triphasic at both temperatures, but for CaS the cell classes are the
same at 10 and 25 C (PD quiescent, LP and PY regular bursting). With the conductance changed from t = 0 (variant), three of
five match: the KCa parts fail (at 10 C, 25% and 0% give PD and PY quiescent with LP tonic; at
25 C, 0% leaves LP tonic) and the CaT and CaS part holds. With a 0 mV spike threshold, 2 (control
first) and 1 (from t = 0) of five match, as LP's spikes in 4X6W66 peak below 0 mV at 10 C (the
intact model has LP quiescent at 0 mV).

## fig10b-complete-removal-differs

> Across the 36 models, the response to complete removal of a current differs between 10 and 25
> degrees C in about half of the models for most currents; most models respond differently for
> CaT, while removing H gives the same state (quiescence) at both temperatures.

All 38 deposited models, each current at 0% at 10 and 25 degrees C, the perturbed run 20 s with
the first 10 s dropped, under both protocols of fig10a-graded-removal (control first, the main
one; from t = 0, a variant).

In [ ]:
t0 = time.time()
G0 = {}
for start in ("control first", "from t = 0"):
    y0 = (lambda n, T: CTRL[(n, T)]) if start == "control first" else (lambda n, T: CI)
    tasks = [((n, k, T), P.scaled(MODELS[n], k, 0.0), y0(n, T), T, 20.0, 10.0) for n in NAMES for k in range(8) for T in (10.0, 25.0)]
    G0[start] = {r["key"]: r["classes"] for r in P.pmap(P.run_fixed, tasks)}
print(f"{2 * len(tasks)} runs in {time.time() - t0:.0f} s")

def table10b(g0):
    rows = []
    for k, cur in enumerate(P.CURRENTS):
        diff_a = np.mean([g0[(n, k, 10.0)] != g0[(n, k, 25.0)] for n in NAMES])
        diff_b = np.mean([g0[(n, k, 10.0)][3] != g0[(n, k, 25.0)][3] for n in NAMES])
        sameq = sum(g0[(n, k, 10.0)] == g0[(n, k, 25.0)] and all(x == "quiescent" for x in g0[(n, k, 10.0)][:3]) for n in NAMES)
        rows.append(dict(current=cur, differ_all_classes=diff_a, differ_NET=diff_b, same_and_quiescent=sameq))
    return pd.DataFrame(rows).set_index("current")

def judge10b(t10b, col, currents):
    t = t10b.loc[currents]
    others = [c for c in currents if c not in ("CaT", "H")]
    cat = t[col].idxmax() == "CaT" and t.loc["CaT", col] > 0.5
    h = t.loc["H", "same_and_quiescent"] >= 33
    half = np.mean([0.3 <= t.loc[c, col] <= 0.7 for c in others])
    return cat, h, half

T10B = {}
for start, g0 in G0.items():
    t10b = T10B[start] = table10b(g0)
    print(f"\n=== protocol ({start})")
    print(t10b.round(3).to_string())
    print("classes after complete removal of H:", Counter(tuple(g0[(n, 6, T)]) for n in NAMES for T in (10.0, 25.0)).most_common(5))
    for col in ("differ_all_classes", "differ_NET"):
        for label, cur in (("8 currents", P.CURRENTS), ("7 without leak", P.CURRENTS[:7])):
            cat, h, half = judge10b(t10b, col, cur)
            print(f"reading {col}, {label}: CaT largest and > 0.5 {cat}; H same and quiescent in >= 33 {h}; "
                  f"fraction of other currents in [0.3, 0.7] {half:.2f}")
g0 = G0["control first"]

**Result: not reproduced** (under both protocols). With the control period first (main
protocol), CaT has the largest fraction of models that respond differently at 10 and 25 C under
both readings (0.895 all classes, 0.605 network class), and 5 of 6 other currents (8 currents) or 4
of 5 (without leak) fall within 0.3 to 0.7 under the four-class reading. But complete removal of H
gives the same, quiescent, state at both temperatures in 30 of 38 models, below the criterion's 33.
With the conductance removed from t = 0 (variant), H gives 31, CaT is the largest only under the
four-class reading (0.974; network class 0.474 against A 0.579), and 2 of 6 or 2 of 5 other
currents fall within 0.3 to 0.7. The H part fails under every reading and protocol; with 36 models
in place of 38, 30 or 31 cannot reach 33.

## Spike threshold at 0 mV (variant)

Every model at 10 and 25 degrees C, 20 s with the first 10 s dropped, with spikes detected at
0 mV in place of -20 mV, and FGFKQS at 30 degrees C (fig6-crash-different-mechanisms). The cell
classes are compared with those at -20 mV from the same runs (deposit-models and the control
runs of fig10a-graded-removal; the 40 s run at 30 degrees C).

In [ ]:
ref = {(r["key"], 10.0): r for r in r10}
ref.update({(r["key"], 25.0): r for r in ctl25})
z = {(r["key"], r["temp"]): r for r in P.pmap(P.run_fixed, [(n, MODELS[n], CI, T, 20.0, 10.0, 0.0) for n in NAMES for T in (10.0, 25.0)])}
changed = [(n, T, ref[(n, T)]["classes"], z[(n, T)]["classes"]) for n, T in ref if ref[(n, T)]["classes"] != z[(n, T)]["classes"]]
fewer = [(n, T, c, len(ref[(n, T)]["spikes"][c]), len(z[(n, T)]["spikes"][c])) for n, T in ref for c in range(3)
         if len(z[(n, T)]["spikes"][c]) != len(ref[(n, T)]["spikes"][c])]
print(f"runs whose classes differ between the -20 and 0 mV thresholds: {len(changed)} of {len(ref)}")
for row in changed:
    print("   ", row)
print(f"cells with a different spike count: {len(fewer)} of {3 * len(ref)}; largest differences (model, T, cell, -20 mV, 0 mV):",
      sorted(fewer, key=lambda x: x[3] - x[4])[-5:])
print("network triphasic at both temperatures, -20 mV:", sum(ref[(n, 10.0)]["classes"][3] == "triphasic" and ref[(n, 25.0)]["classes"][3] == "triphasic" for n in NAMES),
      "; 0 mV:", sum(z[(n, 10.0)]["classes"][3] == "triphasic" and z[(n, 25.0)]["classes"][3] == "triphasic" for n in NAMES), "of", len(NAMES))
f30 = P.run_fixed(("FGFKQS", MODELS["FGFKQS"], CI, 30.0, 40.0, 10.0, 0.0))
print("FGFKQS at 30 C, 0 mV threshold: classes", f30["classes"], "; PD spikes in 30 s", len(f30["spikes"][0]))

## Summary

| Claim | Outcome |
|---|---|
| deposit-models | not reproduced |
| fig1c-frequency-q10-phase-constant | partially reproduced |
| fig1s1-three-models | not reproduced |
| all-models-compensated | partially reproduced |
| intermediate-temperature-failures | blocked (data_missing) |
| figs2-ramp-largest-isi | reproduced (ramps notebook) |
| figs3-duty-cycle-variable-across-models | reproduced, variant reading |
| fig2-lp-spike-amplitude-threshold | reproduced, threshold reading b assumed (voltage_distributions notebook) |
| fig2s1-peak-voltage-diverse | reproduced (voltage_distributions notebook) |
| fig3-current-amplitudes | partially reproduced |
| fig4-currentscapes-reorganize | not reproduced |
| currentscapes-dominant-currents | reproduced |
| fig5-a-to-kca-tradeoff | reproduced |
| fig6-crash-different-mechanisms | not reproduced |
| fig6s1-crash-mechanisms | blocked (data_missing; 7 of 8 models tested, the missing model could make it partial) |
| fig7-high-temperature-slow-timescales | blocked (data_missing) |
| fig8a-hysteresis-quiescence | reproduced (ramps notebook) |
| fig8-up-down-ramps | blocked (too_large; first reading 22 of 38) |
| fig8b-pathological-attractor | not reproduced (ramps notebook) |
| fig9a-remove-a-current | partially reproduced |
| fig9b-remove-kca-current | reproduced |
| fig10a-graded-removal | partially reproduced |
| fig10b-complete-removal-differs | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the models are simulated with a numba port of the
deposited Java, which matches it to 3e-9 mV over 3 s at 18 C, to 0.06 mV over 3 s at 25 C, and
gives NaN at the same step as the Java where the integration diverges (G7P2WE at 35 C, J9U8SQ at
34 C); spikes are upward crossings of -20 mV and the network order is judged on median phases
(assumptions, as the paper gives neither; a 0 mV threshold misses the spikes of cells that peak
below 0 mV and leaves 17 of 38 models triphasic at both 10 and 25 C, against 38); conductances are
changed after a 20 s control period for Figure 10, as the Results do for Figure 9 (assumed; the
change from t = 0 is shown too); the 38 deposited models are used where the paper says 36; the
figure supplements' legends were not read.

What was found about the dataset: 38 models where the paper says 36; 4 models named in the Methods
absent (Q8FBN1, G81PUL, G7EOZ8, SAXT8Y), 3 filed under other spellings and 1 matched at the
character the text cannot render; 27 distinct sets of
maximal conductances; gNa, gA, gKd, the synaptic conductances and tau_Ca (as the Java uses it)
outside the search space the Methods give, in most or all models; the models' burst frequency at
10 C is 1.25 Hz on average, not the 1 Hz target; and the deposited integrator, the Java as well as
the port, diverges (NaN) for 6 models at some temperatures from 26 to 35 C.

In [ ]:
print(f"notebook run time {(time.time() - T_START) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")